In [14]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
username = "aacuser"
password = "verySecurePassword"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('OnlyDogs.com'))),
    
    # Branding
    html.Center([
        html.A(
            html.Img(
                src='data:image/png;base64,{}'.format(encoded_image.decode()),
                style={'width': '200px'}    
                    ),
            href="http://www.snhu.edu"
        ),
        html.P("Dashboard by: Cody Stuart")
    ]),
    html.Hr(),
    html.Div(
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster Rescue or Individual Tracking', 'value': 'disaster'},
                {'label': 'Reset', 'value': 'reset'}
            ],
            value='reset',
            labelStyle={'display': 'inline-block', 'margin-right': '10px'}
        )

    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                        editable=False, # Prevents users from making edits to the table
                        filter_action="native", # Enables a filter box for each column
                        sort_action="native", # Allows users to sort columns by clicking headers
                        sort_mode="multi", # Allows multiple columns to be sorted
                        column_selectable=False, # Disabled the selection of entire columns
                        row_selectable="single", # Allows users to select a single row at a time
                        row_deletable=False, # prevents users from deleting rows
                        selected_columns=[], # Inititalizes wtih no columns selected
                        selected_rows=[0], # Selects the first row by default to ensure the map has initial data
                        page_action="native", # Enables pagination
                        page_current=0, # Starts on the first page
                        page_size=10 # Displays 10 rows per page
                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            ),
        html.Div(
            id='hist-id',
            className='col s12 m6'
        )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    # Check which filter was selected and call the appropriate CRUD method
    if filter_type == 'water':
        data = db.read_water_rescue()
    elif filter_type == 'mountain':
        data = db.read_mountain_rescue()
    elif filter_type == 'disaster':
        data = db.read_disaster_rescue()
    else:
        data = db.read({}) # 'reset' or default case

    # If no data is returned, return an empty list
    if not data:
        return []
    
    # Convert the list of dicts from MongoDB into a DataFrame
    df = pd.DataFrame.from_records(data)
    
    # MongoDB v5+ is going to return the '_id' column and that is going to have an 
    # invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
    # it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
    # inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
    df.drop(columns=['_id'], inplace=True)
    
    # Return the filtered data as a list of dictionaries
    return df.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    # If the table data is empty, return nothing
    if not viewData:
        return []
    # Convert the visible table data into a DataFram
    dff = pd.DataFrame.from_dict(viewData)
    
    # Get the counts of each breed for the pie chart
    pie_data = dff['breed'].value_counts().reset_index()
    pie_data.columns = ['breed', 'count']
    
    # Create the pie cahrt figure
    fig_pie = px.pie(pie_data,
                     names='breed',
                     values='count',
                     title='Preferred Animals Breed Shares')

    return [
        dcc.Graph(
            figure=fig_pie,
            responsive=False
        )
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    
    if dff.empty:
        return [
                dl.Map(style={'width': '1000px', 'height': '500px'},
                   center=[30.75,-97.48], zoom=10, children=[
                   dl.TileLayer(id="base-layer-id")
                   ])
                ]   
    # Because we only allow single row selection, the list can be converted to a row index here
    if not index:
        row = 0
    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, 
            center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], 
                children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]

# Callback to create and display the Histogram
@app.callback(
    Output('hist-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_histogram(viewData):
    # if the table is empty, return nothing
    if not viewData:
        return []
    
    # Convert the visibale table data into a DataFrame
    dff = pd.DataFrame.from_dict(viewData)

    # Create the histogram figure using the age in weeks column
    fig_hist = px.histogram(dff,
                            x='age_upon_outcome_in_weeks',
                            title='Age Distribution (in weeks)')

    return [
        dcc.Graph(
            figure=fig_hist,
            responsive=False
        )
    ]
# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Connection to MongoDB successful!
Dash app running on https://palacejunior-sourcecontrol-3000.codio.io/proxy/8050/
